# Seattle weather — quick look

Just poking at some hourly temperature data for SeaTac.
Want to see the seasonal + daily pattern and try a naive next-day forecast.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# generate a year of hourly data for SeaTac
# (in a real analysis this would be a CSV from NOAA, but this makes the
# notebook self-contained for the demo)

np.random.seed(42)

hours = pd.date_range('2024-01-01 00:00', '2024-12-31 23:00', freq='h', tz='UTC')

# seasonal signal: cold in Jan/Feb, warm in Jul/Aug
doy = hours.dayofyear.to_numpy()
seasonal = 9 + 11 * np.sin(2 * np.pi * (doy - 100) / 365.25)

# diurnal signal: peak at ~22:00 UTC = ~15:00 PST (3 PM, the realistic afternoon peak)
hod = hours.hour.to_numpy()
diurnal = 5 * np.sin(2 * np.pi * (hod - 16) / 24)

# noise
noise = np.random.normal(0, 1.8, len(hours))

temp = seasonal + diurnal + noise

df = pd.DataFrame({
    'timestamp': hours.tz_localize(None),  # strip tz — data files usually don't carry it
    'temp_c': temp,
})

df.to_csv('ksea_2024.csv', index=False)
print(f'wrote {len(df)} rows')

## Load the data

In [ ]:
df = pd.read_csv('ksea_2024.csv')
df.head()

In [ ]:
df.describe()

## Clean up

Convert the timestamp column and set as index.

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp')
df.head()

In [ ]:
df['temp_c'].plot(figsize=(12, 3))
plt.ylabel('temperature (C)')
plt.title('Hourly temperature, 2024')
plt.show()

## Daily aggregation

Get daily mean/min/max.

In [ ]:
daily = df['temp_c'].resample('D').agg(['mean', 'max', 'min'])
daily.head(10)

In [ ]:
daily[['max', 'mean', 'min']].plot(figsize=(12, 4))
plt.ylabel('temperature (C)')
plt.title('Daily temperature, 2024')
plt.show()

## Diurnal pattern

What does a typical day look like?

In [ ]:
df['hour'] = df.index.hour
by_hour = df.groupby('hour')['temp_c'].mean()
by_hour.plot(kind='bar', figsize=(10, 3))
plt.ylabel('mean temp (C)')
plt.xlabel('hour of day')
plt.title('Average temperature by hour of day')
plt.show()

print(f'peak hour: {by_hour.idxmax()}')
print(f'coldest hour: {by_hour.idxmin()}')

## Naive forecast

Predict tomorrow's max as the mean of the last 7 daily maxes.

In [ ]:
last_7 = daily['max'].iloc[-7:]
forecast = last_7.mean()
print(f'last 7 daily maxes:')
print(last_7)
print(f'\nforecast for next day max: {forecast:.1f} C')